# WildReceipt Benchmark — InternVL3.5-8B (vLLM Data-Parallel)

**WildReceipt: Receipt Key Information Extraction**

This notebook benchmarks [InternVL3.5-8B](https://huggingface.co/OpenGVLab/InternVL3-8B) on the
[WildReceipt](https://download.openmmlab.com/mmocr/data/wildreceipt.tar) dataset using **vLLM**
with **data parallelism** — one independent engine per GPU, images partitioned round-robin
across workers. Scales linearly with GPU count.

| Field | Type | Description |
|-------|------|-------------|
| **Store_name** | scalar | Store / business name |
| **Store_addr** | scalar | Full store address |
| **Tel** | scalar | Telephone number |
| **Date** | scalar | Receipt date (DD/MM/YYYY) |
| **Time** | scalar | Receipt time (HH:MM) |
| **Subtotal** | scalar | Subtotal before tax |
| **Tax** | scalar | Tax amount |
| **Tips** | scalar | Tips amount |
| **Total** | scalar | Final total amount |
| **Prod_item** | list | Product item names (pipe-delimited) |
| **Prod_quantity** | list | Product quantities (pipe-delimited) |
| **Prod_price** | list | Product prices (pipe-delimited) |

**Metrics:** Per-field precision, recall, and F1 (entity-level exact match after normalization),
plus overall F1 (mean of per-field F1 across fields with at least one occurrence).

**Self-contained:** All evaluation, preprocessing, and inference logic is inlined.
Requires `vllm`, `Pillow`, `tqdm`, and `pyyaml`.

---
## 1. Configuration

In [1]:
from pathlib import Path

# --- Paths ---
DATA_DIR = Path("data/wildreceipt")  # Created automatically by the download cell below
MODEL_PATH = Path("/home/jovyan/nfs_share/models/InternVL3_5-8B")

# --- Data-parallel settings ---
NUM_GPUS = 2  # Number of DP workers (one vLLM engine per GPU)
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 8192

# --- Inference settings ---
MAX_IMAGES = None  # Set to an int (e.g. 10) for quick test runs
MAX_NEW_TOKENS = 512  # Higher than SROIE (256) for list fields

# --- Comparison notebook output paths ---
COMPARISON_GT_PATH = Path("evaluation_data/wildreceipt_ground_truth.yaml")
COMPARISON_RESULTS_PATH = Path("results/wildreceipt_internvl3_5_8b_results.yaml")

---
## 2. Dataset Setup

The WildReceipt dataset is downloaded automatically from OpenMMLab into
`data/wildreceipt/` (relative to this notebook) if not already present.

```
data/wildreceipt/
  image_files/     # Receipt images (.jpeg) in subdirectories
  train.txt        # Training annotations (JSON lines)
  test.txt         # Test annotations (JSON lines)
  class_list.txt   # NER label definitions
```

Each line in `test.txt` is a JSON object with `file_name`, `height`, `width`, and
`annotations` — a list of bounding boxes with `text` and NER `label` IDs.

In [2]:
import shutil
import subprocess
import sys

DATA_DIR.mkdir(parents=True, exist_ok=True)

test_file = DATA_DIR / "test.txt"
image_dir = DATA_DIR / "image_files"

if not test_file.exists() or not image_dir.exists() or not any(image_dir.rglob("*.jpeg")):
    print("WildReceipt dataset not found — downloading ...")
    tar_path = DATA_DIR / "wildreceipt.tar"
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            str(tar_path),
            "https://download.openmmlab.com/mmocr/data/wildreceipt.tar",
        ],
        check=True,
    )
    subprocess.run(["tar", "xf", str(tar_path), "-C", str(DATA_DIR)], check=True)
    tar_path.unlink()
    # Handle nested directory: tar extracts wildreceipt/ inside DATA_DIR.
    # Remove any stale dest dirs/files before moving to avoid skipping the rename.
    nested = DATA_DIR / "wildreceipt"
    if nested.exists() and nested.is_dir():
        for child in nested.iterdir():
            dest = DATA_DIR / child.name
            if dest.exists():
                if dest.is_dir():
                    shutil.rmtree(dest)
                else:
                    dest.unlink()
            child.rename(dest)
        shutil.rmtree(nested)
    print("Download complete.")

assert test_file.exists(), f"Test file not found: {test_file}"
assert image_dir.exists(), f"Image directory not found: {image_dir}"
n_images = len(list(image_dir.rglob("*.jpeg")))
print(f"Test file: {test_file}")
print(f"Image directory: {image_dir} ({n_images} images)")

WildReceipt dataset not found — downloading ...
Download complete.
Test file: data/wildreceipt/test.txt
Image directory: data/wildreceipt/image_files (1765 images)


---
## 3. Imports

In [3]:
import gc
import json
import re
import subprocess
import sys
import time
from dataclasses import dataclass, field

import yaml
from PIL import Image
from tqdm.auto import tqdm

---
## 4. Ground Truth Loading

WildReceipt annotations use NER labels where even IDs are `_key` labels and odd IDs
are `_value` labels. We extract only the `_value` labels (odd IDs 1–23), mapping them
to 12 fields. Tokens sharing the same label within an image are concatenated with spaces.

In [4]:
# --- NER label ID → field name mapping (odd IDs = value labels) ---
LABEL_TO_FIELD = {
    1: "Store_name",
    3: "Store_addr",
    5: "Tel",
    7: "Date",
    9: "Time",
    11: "Prod_item",
    13: "Prod_quantity",
    15: "Prod_price",
    17: "Subtotal",
    19: "Tax",
    21: "Tips",
    23: "Total",
}

WILDRECEIPT_FIELDS = (
    "Store_name",
    "Store_addr",
    "Tel",
    "Date",
    "Time",
    "Prod_item",
    "Prod_quantity",
    "Prod_price",
    "Subtotal",
    "Tax",
    "Tips",
    "Total",
)

SCALAR_FIELDS = {
    "Store_name",
    "Store_addr",
    "Tel",
    "Date",
    "Time",
    "Subtotal",
    "Tax",
    "Tips",
    "Total",
}
LIST_FIELDS = {"Prod_item", "Prod_quantity", "Prod_price"}


def load_wildreceipt_ground_truth(
    test_path: Path,
) -> tuple[dict[str, dict[str, str]], dict[str, str]]:
    """Parse WildReceipt test.txt and aggregate tokens by NER label.

    Returns:
        ground_truth: {image_id: {field_name: concatenated_value}}
        image_id_to_relpath: {image_id: relative file path from test.txt}
    """
    ground_truth: dict[str, dict[str, str]] = {}
    image_id_to_relpath: dict[str, str] = {}
    stem_counts: dict[str, int] = {}

    entries = []
    with test_path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            entries.append(entry)
            stem = Path(entry["file_name"]).stem
            stem_counts[stem] = stem_counts.get(stem, 0) + 1

    # Check if stems are unique; use composite key if not
    duplicates = {s for s, c in stem_counts.items() if c > 1}
    use_composite = len(duplicates) > 0
    if use_composite:
        print(f"Warning: {len(duplicates)} duplicate stems found, using composite keys")

    for entry in entries:
        file_name = entry["file_name"]
        p = Path(file_name)
        image_id = f"{p.parent.name}/{p.stem}" if use_composite else p.stem

        # Aggregate tokens by field
        field_tokens: dict[str, list[str]] = {}
        for ann in entry.get("annotations", []):
            label = ann["label"]
            if label in LABEL_TO_FIELD:
                field_name = LABEL_TO_FIELD[label]
                text = ann.get("text", "").strip()
                if text:
                    field_tokens.setdefault(field_name, []).append(text)

        gt_entry = {}
        for field_name, tokens in field_tokens.items():
            gt_entry[field_name] = " ".join(tokens)

        ground_truth[image_id] = gt_entry
        image_id_to_relpath[image_id] = file_name

    return ground_truth, image_id_to_relpath


ground_truth, image_id_to_relpath = load_wildreceipt_ground_truth(test_file)

# Verify image ID uniqueness
assert len(ground_truth) == len(image_id_to_relpath), (
    f"Duplicate image IDs: {len(image_id_to_relpath)} paths vs "
    f"{len(ground_truth)} unique IDs"
)

# Resolve absolute paths and verify images exist
all_images: list[tuple[str, Path]] = []
missing = 0
for image_id in sorted(ground_truth):
    rel_path = image_id_to_relpath[image_id]
    abs_path = DATA_DIR / rel_path
    if abs_path.exists():
        all_images.append((image_id, abs_path))
    else:
        missing += 1

if missing:
    print(f"Warning: {missing} images not found on disk")

print(f"Loaded {len(ground_truth)} ground truth entries")
print(f"Images on disk: {len(all_images)}")
print()

# Show field coverage
for f in WILDRECEIPT_FIELDS:
    count = sum(1 for gt in ground_truth.values() if f in gt)
    print(f"  {f:<15} {count:>4} images")

Loaded 472 ground truth entries
Images on disk: 472

  Store_name       472 images
  Store_addr       366 images
  Tel              296 images
  Date             398 images
  Time             387 images
  Prod_item        446 images
  Prod_quantity    331 images
  Prod_price       446 images
  Subtotal         343 images
  Tax              318 images
  Tips              21 images
  Total            422 images


---
## 5. Evaluation Functions

In [5]:
# --- Result dataclasses ---


@dataclass
class WildReceiptFieldResult:
    """Aggregate metrics for one WildReceipt field across all images."""

    field: str
    true_positives: int = 0
    false_positives: int = 0
    false_negatives: int = 0

    @property
    def precision(self) -> float:
        denom = self.true_positives + self.false_positives
        return self.true_positives / denom if denom > 0 else 0.0

    @property
    def recall(self) -> float:
        denom = self.true_positives + self.false_negatives
        return self.true_positives / denom if denom > 0 else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    @property
    def total_occurrences(self) -> int:
        return self.true_positives + self.false_positives + self.false_negatives


@dataclass
class WildReceiptImageResult:
    """Per-image extraction result."""

    image_id: str
    ground_truth: dict[str, str]
    predicted: dict[str, str]
    matches: dict[str, bool] = field(default_factory=dict)
    processing_time: float = 0.0


@dataclass
class WildReceiptBenchmarkResult:
    """Full benchmark result for one model run."""

    model_name: str
    image_results: list[WildReceiptImageResult]
    field_results: dict[str, WildReceiptFieldResult]
    overall_f1: float
    total_images: int
    elapsed_seconds: float = 0.0


# --- Normalization ---

_WHITESPACE_RE = re.compile(r"\s+")
_CURRENCY_RE = re.compile(r"[$\u00a3\u20ac\u00a5RM]")
_COMMA_IN_NUMBER_RE = re.compile(r"(\d),(\d)")


def normalize_text(text: str) -> str:
    """General text normalization: lowercase, collapse whitespace, strip."""
    text = text.lower().strip()
    text = _WHITESPACE_RE.sub(" ", text)
    return text


def normalize_total(text: str) -> str:
    """Normalize monetary total: strip currency symbols, commas, whitespace."""
    text = text.strip()
    text = _CURRENCY_RE.sub("", text)
    text = _COMMA_IN_NUMBER_RE.sub(r"\1\2", text)
    text = text.strip()
    try:
        val = float(text)
        return f"{val:.2f}"
    except ValueError:
        return normalize_text(text)


def normalize_date(text: str) -> str:
    """Normalize date to DD/MM/YYYY format."""
    text = text.strip()

    # ISO format: YYYY-MM-DD
    m = re.match(r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})", text)
    if m:
        return f"{int(m.group(3)):02d}/{int(m.group(2)):02d}/{m.group(1)}"

    # DD/MM/YYYY or DD-MM-YYYY or DD.MM.YYYY
    m = re.match(r"(\d{1,2})[-/.](\d{1,2})[-/.](\d{4})", text)
    if m:
        return f"{int(m.group(1)):02d}/{int(m.group(2)):02d}/{m.group(3)}"

    # DD Mon YYYY
    months = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12",
    }
    m = re.match(r"(\d{1,2})\s+(\w{3,})\s+(\d{4})", text, re.IGNORECASE)
    if m:
        month_str = m.group(2)[:3].lower()
        if month_str in months:
            return f"{int(m.group(1)):02d}/{months[month_str]}/{m.group(3)}"

    return normalize_text(text)


def normalize_time(text: str) -> str:
    """Normalize time to HH:MM (24-hour) format."""
    text = text.strip()
    m = re.match(r"(\d{1,2}):(\d{2})(?::\d{2})?\s*(am|pm)?", text, re.IGNORECASE)
    if m:
        hour = int(m.group(1))
        minute = int(m.group(2))
        ampm = m.group(3)
        if ampm:
            if ampm.lower() == "pm" and hour < 12:
                hour += 12
            elif ampm.lower() == "am" and hour == 12:
                hour = 0
        return f"{hour:02d}:{minute:02d}"
    return normalize_text(text)


def normalize_tel(text: str) -> str:
    """Normalize telephone number: digits only."""
    digits = re.sub(r"[^\d]", "", text)
    return digits if digits else normalize_text(text)


# --- Matching ---

_MONETARY_FIELDS = {"Total", "Subtotal", "Tax", "Tips", "Prod_price"}


def wildreceipt_field_match(
    field_name: str, predicted: str, ground_truth_val: str
) -> bool:
    """Check exact match after field-specific normalization."""
    if not predicted or not ground_truth_val:
        return False

    if field_name in _MONETARY_FIELDS:
        return normalize_total(predicted) == normalize_total(ground_truth_val)
    if field_name == "Date":
        return normalize_date(predicted) == normalize_date(ground_truth_val)
    if field_name == "Time":
        return normalize_time(predicted) == normalize_time(ground_truth_val)
    if field_name == "Tel":
        return normalize_tel(predicted) == normalize_tel(ground_truth_val)
    return normalize_text(predicted) == normalize_text(ground_truth_val)


# --- Aggregation ---


def compute_wildreceipt_metrics(
    image_results: list[WildReceiptImageResult],
    model_name: str = "unknown",
    elapsed_seconds: float = 0.0,
) -> WildReceiptBenchmarkResult:
    """Aggregate per-image results into per-field and overall metrics."""
    field_results = {f: WildReceiptFieldResult(field=f) for f in WILDRECEIPT_FIELDS}

    for img_result in image_results:
        for f in WILDRECEIPT_FIELDS:
            gt_val = img_result.ground_truth.get(f, "")
            pred_val = img_result.predicted.get(f, "")

            # Skip fields absent from both GT and prediction for this image
            if not gt_val and not pred_val:
                continue

            match = wildreceipt_field_match(f, pred_val, gt_val)
            img_result.matches[f] = match

            if match:
                field_results[f].true_positives += 1
            else:
                if pred_val:
                    field_results[f].false_positives += 1
                if gt_val:
                    field_results[f].false_negatives += 1

    # Overall F1: mean of per-field F1, only counting fields with at least one occurrence
    f1_scores = [fr.f1 for fr in field_results.values() if fr.total_occurrences > 0]
    overall_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

    return WildReceiptBenchmarkResult(
        model_name=model_name,
        image_results=image_results,
        field_results=field_results,
        overall_f1=overall_f1,
        total_images=len(image_results),
        elapsed_seconds=elapsed_seconds,
    )


print("Evaluation functions loaded.")

Evaluation functions loaded.


---
## 6. Data-Parallel Worker

Each GPU runs an independent vLLM engine (TP=1) in its own subprocess, pinned via
`CUDA_VISIBLE_DEVICES`. Images are partitioned round-robin across workers for load
balance, then results are merged back in original order.

The cell below writes a self-contained worker script to disk.

In [6]:
_WORKER_SCRIPT = DATA_DIR / "_wildreceipt_dp_worker.py"

_WORKER_SCRIPT.write_text(r'''
#!/usr/bin/env python3
"""WildReceipt benchmark DP worker — processes a shard of images on one GPU."""
import argparse
import base64
import io
import json
import os
import sys
import time as _time

from PIL import Image


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--gpu-id", type=int, required=True)
    parser.add_argument("--model-path", required=True)
    parser.add_argument("--image-list", required=True)
    parser.add_argument("--prompt-file", required=True)
    parser.add_argument("--output-file", required=True)
    parser.add_argument("--max-tokens", type=int, default=512)
    parser.add_argument("--max-model-len", type=int, default=8192)
    parser.add_argument("--gpu-mem", type=float, default=0.90)
    args = parser.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu_id)
    os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

    from vllm import LLM, SamplingParams

    print(f"[GPU {args.gpu_id}] Loading model ...", flush=True)
    model = LLM(
        model=args.model_path,
        tensor_parallel_size=1,
        max_model_len=args.max_model_len,
        gpu_memory_utilization=args.gpu_mem,
        max_num_seqs=1,
        limit_mm_per_prompt={"image": 1},
        trust_remote_code=True,
        disable_log_stats=True,
        enforce_eager=False,
        enable_prefix_caching=True,
    )

    sampling = SamplingParams(max_tokens=args.max_tokens, temperature=0)

    with open(args.image_list) as f:
        image_paths = json.load(f)
    with open(args.prompt_file) as f:
        prompt = f.read()

    results = []
    total = len(image_paths)
    print(f"[GPU {args.gpu_id}] Processing {total} images ...", flush=True)

    for idx, img_path in enumerate(image_paths):
        try:
            t0 = _time.time()
            image = Image.open(img_path).convert("RGB")
            buf = io.BytesIO()
            image.save(buf, format="PNG")
            data_uri = (
                f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"
            )
            buf.close()

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": data_uri}},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]

            outputs = model.chat(
                messages=messages, sampling_params=sampling, use_tqdm=False
            )
            text = outputs[0].outputs[0].text.strip()
            elapsed = _time.time() - t0
            del outputs, messages, data_uri, image
            results.append({"image_path": img_path, "response": text, "time": round(elapsed, 3)})
        except Exception as e:
            results.append({"image_path": img_path, "response": "", "error": str(e), "time": 0.0})

        if (idx + 1) % 50 == 0 or idx + 1 == total:
            print(f"[GPU {args.gpu_id}] {idx + 1}/{total}", flush=True)

    with open(args.output_file, "w") as f:
        json.dump(results, f)

    del model
    print(f"[GPU {args.gpu_id}] Done.", flush=True)


if __name__ == "__main__":
    main()
''')
print(f"Worker script: {_WORKER_SCRIPT}")

Worker script: data/wildreceipt/_wildreceipt_dp_worker.py


---
## 7. Response Parsing

In [7]:
_FIELD_RE = re.compile(
    r"^(Store_name|Store_addr|Tel|Date|Time|Prod_item|Prod_quantity|Prod_price|Subtotal|Tax|Tips|Total)\s*:\s*(.+)$",
    re.IGNORECASE | re.MULTILINE,
)

# Map lowercase field names back to canonical capitalization
_CANONICAL_FIELD = {f.lower(): f for f in WILDRECEIPT_FIELDS}


def parse_wildreceipt_response(raw_response: str) -> dict[str, str]:
    """Parse model response into WildReceipt field dict.

    Expects lines like:
        Store_name: SOME STORE NAME
        Date: 25/12/2018
        Total: 4.95
    """
    result: dict[str, str] = {}
    for match in _FIELD_RE.finditer(raw_response):
        field_name = _CANONICAL_FIELD.get(match.group(1).lower())
        if field_name is None:
            continue
        value = match.group(2).strip()
        if value.upper() != "NOT_FOUND":
            result[field_name] = value
    return result

---
## 8. Extraction Prompt

In [8]:
WILDRECEIPT_PROMPT = """Extract the following 12 fields from this receipt image.
Return each field on its own line in exactly this format:

Store_name: <the store or business name>
Store_addr: <the full store address>
Tel: <the telephone number>
Date: <the receipt date in DD/MM/YYYY format>
Time: <the time in HH:MM (24-hour) format>
Prod_item: <product names separated by | e.g. Coffee|Tea|Cake>
Prod_quantity: <quantities separated by | matching product order e.g. 1|2|1>
Prod_price: <prices separated by | matching product order, no currency symbol e.g. 3.50|4.00|5.25>
Subtotal: <the subtotal amount as a number, no currency symbol>
Tax: <the tax amount as a number, no currency symbol>
Tips: <the tips amount as a number, no currency symbol>
Total: <the final total amount as a number, no currency symbol>

Rules:
- For product fields (Prod_item, Prod_quantity, Prod_price), list each item separated by | in order.
- For "Date", convert to DD/MM/YYYY format regardless of how it appears.
- For "Time", convert to HH:MM (24-hour) format.
- For monetary fields (Subtotal, Tax, Tips, Total, Prod_price), use only numbers (no currency symbol).
- If a field is not visible, write NOT_FOUND.
- Do NOT add any extra text, explanations, or formatting.
"""

print(f"Prompt length: {len(WILDRECEIPT_PROMPT)} chars")

Prompt length: 1226 chars


---
## 9. Benchmark Loop

In [9]:
# Apply MAX_IMAGES limit if set
benchmark_images = all_images[:MAX_IMAGES] if MAX_IMAGES else all_images

# Partition images across GPUs (round-robin for load balance)
shards: list[list[str]] = [[] for _ in range(NUM_GPUS)]
shard_ids: list[list[str]] = [[] for _ in range(NUM_GPUS)]  # Track image_ids per shard
for idx, (image_id, img_path) in enumerate(benchmark_images):
    gpu = idx % NUM_GPUS
    shards[gpu].append(str(img_path))
    shard_ids[gpu].append(image_id)

# Write prompt and shard manifests
prompt_file = DATA_DIR / "_prompt.txt"
prompt_file.write_text(WILDRECEIPT_PROMPT)

print(f"Running benchmark: {len(benchmark_images)} images across {NUM_GPUS} GPUs")
for i, shard in enumerate(shards):
    print(f"  GPU {i}: {len(shard)} images")

# Build worker environment: prepend the active Python environment's lib/ so the
# subprocess finds the correct libstdc++ instead of the base conda copy.
# Derive from sys.executable (always correct) rather than CONDA_PREFIX (may point
# to the base env instead of the active env).
import os
import selectors

worker_env = os.environ.copy()
env_lib = str(Path(sys.executable).resolve().parent.parent / "lib")
ld_path = worker_env.get("LD_LIBRARY_PATH", "")
if env_lib not in ld_path:
    worker_env["LD_LIBRARY_PATH"] = env_lib + (":" + ld_path if ld_path else "")
print(f"  LD_LIBRARY_PATH = {worker_env['LD_LIBRARY_PATH']}")

# Launch DP workers
start_time = time.time()
procs = []
output_files = []

for gpu_id in range(NUM_GPUS):
    shard_file = DATA_DIR / f"_shard_{gpu_id}.json"
    output_file = DATA_DIR / f"_results_{gpu_id}.json"
    shard_file.write_text(json.dumps(shards[gpu_id]))
    output_files.append(output_file)

    cmd = [
        sys.executable,
        str(_WORKER_SCRIPT),
        "--gpu-id",
        str(gpu_id),
        "--model-path",
        str(MODEL_PATH),
        "--image-list",
        str(shard_file),
        "--prompt-file",
        str(prompt_file),
        "--output-file",
        str(output_file),
        "--max-tokens",
        str(MAX_NEW_TOKENS),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-mem",
        str(GPU_MEMORY_UTILIZATION),
    ]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=worker_env,
    )
    procs.append(proc)

# Stream worker output in real time from the main thread (Jupyter requirement).
# Uses selectors to multiplex reads from all worker pipes concurrently.
sel = selectors.DefaultSelector()
for gpu_id, proc in enumerate(procs):
    sel.register(proc.stdout, selectors.EVENT_READ, gpu_id)

active = len(procs)
while active > 0:
    for key, _ in sel.select(timeout=1.0):
        line = key.fileobj.readline()
        if line:
            print(line, end="", flush=True)
        else:
            sel.unregister(key.fileobj)
            active -= 1

failed_gpus = []
for gpu_id, proc in enumerate(procs):
    proc.wait()
    if proc.returncode != 0:
        print(f"ERROR: Worker GPU {gpu_id} failed (exit {proc.returncode})")
        failed_gpus.append(gpu_id)

elapsed = time.time() - start_time

if failed_gpus:
    raise RuntimeError(
        f"Workers on GPU(s) {failed_gpus} failed — no result files produced. "
        f"Check the worker output above for details (common cause: libstdc++ version mismatch)."
    )

# Merge results from all workers
response_map: dict[str, dict] = {}  # image_path -> {response, time}
for output_file in output_files:
    with output_file.open() as f:
        for r in json.load(f):
            response_map[r["image_path"]] = {
                "response": r["response"],
                "time": r.get("time", 0.0),
            }

# Build image_results in original order
image_results: list[WildReceiptImageResult] = []
for image_id, img_path in benchmark_images:
    gt = ground_truth[image_id]
    worker_result = response_map.get(str(img_path), {"response": "", "time": 0.0})
    raw_response = worker_result["response"]
    predicted = parse_wildreceipt_response(raw_response)
    image_results.append(
        WildReceiptImageResult(
            image_id=image_id,
            ground_truth=gt,
            predicted=predicted,
            processing_time=worker_result["time"],
        )
    )

# Cleanup temp files
for gpu_id in range(NUM_GPUS):
    (DATA_DIR / f"_shard_{gpu_id}.json").unlink(missing_ok=True)
    (DATA_DIR / f"_results_{gpu_id}.json").unlink(missing_ok=True)
prompt_file.unlink(missing_ok=True)

print(
    f"\nDone. {len(image_results)} images in {elapsed:.1f}s "
    f"({len(image_results) / elapsed * 60:.1f} images/min)"
)

Running benchmark: 472 images across 2 GPUs
  GPU 0: 236 images
  GPU 1: 236 images
  LD_LIBRARY_PATH = /home/jovyan/.conda/envs/vllm_env2/lib:/usr/local/cuda-12.4/lib64:
[GPU 1] Loading model ...
INFO 05-27 03:37:44 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enable_prefix_caching': True, 'max_num_seqs': 1, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 1}, 'model': '/home/jovyan/nfs_share/models/InternVL3_5-8B'}
[GPU 0] Loading model ...
INFO 05-27 03:37:44 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enable_prefix_caching': True, 'max_num_seqs': 1, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 1}, 'model': '/home/jovyan/nfs_share/models/InternVL3_5-8B'}
INFO 05-27 03:37:44 [model.py:549] Resolved architecture: InternVLChatModel
INFO 05-27 03:37:44 [model.py:1678] Using max model len 8192
INFO 05-27 03:37:44 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tok

---
## 10. Results

In [10]:
result = compute_wildreceipt_metrics(image_results, "InternVL3.5-8B", elapsed)

# --- Per-field table ---
print(
    f"{'Field':<15} {'Precision':>10} {'Recall':>10} {'F1':>10}  {'TP':>5} {'FP':>5} {'FN':>5}"
)
print("-" * 70)
for f in WILDRECEIPT_FIELDS:
    fr = result.field_results[f]
    if fr.total_occurrences == 0:
        print(
            f"{f:<15} {'N/A':>10} {'N/A':>10} {'N/A':>10}"
            f"  {fr.true_positives:>5} {fr.false_positives:>5} {fr.false_negatives:>5}"
        )
    else:
        print(
            f"{f:<15} {fr.precision * 100:>9.1f}% {fr.recall * 100:>9.1f}% {fr.f1 * 100:>9.1f}%"
            f"  {fr.true_positives:>5} {fr.false_positives:>5} {fr.false_negatives:>5}"
        )
print("-" * 70)
print(f"{'Overall F1':<15} {'':>10} {'':>10} {result.overall_f1 * 100:>9.1f}%")

# --- Throughput ---
print(f"\nImages: {result.total_images}")
print(f"Time:   {result.elapsed_seconds:.1f}s")
if result.elapsed_seconds > 0:
    print(f"Speed:  {result.total_images / result.elapsed_seconds * 60:.1f} images/min")

Field            Precision     Recall         F1     TP    FP    FN
----------------------------------------------------------------------
Store_name           12.3%      12.3%      12.3%     58   413   414
Store_addr            0.0%       0.0%       0.0%      0   407   366
Tel                  76.0%      85.5%      80.4%    253    80    43
Date                 26.3%      27.4%      26.8%    109   306   289
Time                 78.5%      80.1%      79.3%    310    85    77
Prod_item             2.2%       2.2%       2.2%     10   441   436
Prod_quantity         6.2%       8.5%       7.2%     28   421   303
Prod_price            9.6%       9.6%       9.6%     43   405   403
Subtotal             59.6%      76.7%      67.1%    263   178    80
Tax                  64.0%      71.1%      67.4%    226   127    92
Tips                 18.2%      66.7%      28.6%     14    63     7
Total                52.8%      57.3%      55.0%    242   216   180
---------------------------------------------

---
## 11. Error Analysis

In [11]:
# Show mismatched predictions for debugging
errors = [
    img
    for img in result.image_results
    if not all(
        img.matches.get(f, False) for f in WILDRECEIPT_FIELDS if f in img.ground_truth
    )
]

print(f"{len(errors)} images with at least one field mismatch:\n")

for img in errors[:20]:  # Show first 20
    mismatched = [
        f
        for f in WILDRECEIPT_FIELDS
        if f in img.ground_truth and not img.matches.get(f, False)
    ]
    if not mismatched:
        continue
    print(f"--- {img.image_id} (mismatched: {', '.join(mismatched)}) ---")
    for f in mismatched:
        gt_val = img.ground_truth.get(f, "")
        pred_val = img.predicted.get(f, "<missing>")
        print(f"  {f:<15} GT:   {gt_val}")
        print(f"  {'':>15} PRED: {pred_val}")
    print()

471 images with at least one field mismatch:

--- 018af7323503309c0b84d729a28f32df63e396e9 (mismatched: Store_addr, Tel, Date, Prod_item, Prod_quantity, Prod_price, Tax, Total) ---
  Store_addr      GT:   UNITOF:FUTOMICDINERSLLP.
                  PRED: Unit of: Futomic Diners LLP, 011-45026500, 011-45026800 Connaught Place, Delhi
  Tel             GT:   011-45026500,011-45026800
                  PRED: 07820040373
  Date            GT:   Tuesday,March27th2018
                  PRED: 27/03/2018
  Prod_item       GT:   Platter AssortedPastrias Nos-Kingoffruits Nos-ChocolateOreo Nos-AppleMojitos JungleeHandtKukkad JalpartFishTikka
                  PRED: Chicken Dum Biryani|Jalapati Fish Tikka|Jungleee Handi Kukkad|Nos - Apple Mojitos|Nos - Chocolate Oreo|Nos - King of fruits|Assorted Pastries Platter
  Prod_quantity   GT:   1 1 2 1 1 1 2
                  PRED: 2|1|1|2|2|1|1
  Prod_price      GT:   750.00 475.00 425.00 125.00 300.00 150.00 199.00 Guests:4
                  PRED: 750|475

---
## 12. Export Results

In [12]:
import csv

output_dir = DATA_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# --- Per-image CSV ---
csv_path = output_dir / "wildreceipt_internvl3_per_image.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    header = ["image_id"]
    for field_name in WILDRECEIPT_FIELDS:
        header.extend([f"{field_name}_gt", f"{field_name}_pred", f"{field_name}_match"])
    header.append("processing_time")
    writer.writerow(header)

    for img in result.image_results:
        row = [img.image_id]
        for field_name in WILDRECEIPT_FIELDS:
            row.extend(
                [
                    img.ground_truth.get(field_name, ""),
                    img.predicted.get(field_name, ""),
                    img.matches.get(field_name, False),
                ]
            )
        row.append(img.processing_time)
        writer.writerow(row)

print(f"Per-image results saved to {csv_path}")

# --- Summary JSON ---
summary = {
    "model": result.model_name,
    "overall_f1": round(result.overall_f1, 4),
    "total_images": result.total_images,
    "elapsed_seconds": round(result.elapsed_seconds, 2),
    "per_field": {
        f: {
            "precision": round(result.field_results[f].precision, 4),
            "recall": round(result.field_results[f].recall, 4),
            "f1": round(result.field_results[f].f1, 4),
            "occurrences": result.field_results[f].total_occurrences,
        }
        for f in WILDRECEIPT_FIELDS
    },
}
json_path = output_dir / "wildreceipt_internvl3_summary.json"
json_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {json_path}")

# --- Ground truth YAML (for comparison notebook) ---
gt_yaml = {
    "images": {
        img.image_id: {k: v for k, v in img.ground_truth.items() if v}
        for img in result.image_results
    }
}
COMPARISON_GT_PATH.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_GT_PATH.write_text(
    yaml.dump(gt_yaml, default_flow_style=False, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)
print(f"Ground truth YAML saved to {COMPARISON_GT_PATH}")

# --- Results YAML (for comparison notebook) ---
results_yaml = {
    "results": [
        {
            "image": img.image_id,
            "processing_time": round(img.processing_time, 3),
            "fields": {k: v for k, v in img.predicted.items() if v},
        }
        for img in result.image_results
    ]
}
COMPARISON_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_RESULTS_PATH.write_text(
    yaml.dump(
        results_yaml, default_flow_style=False, allow_unicode=True, sort_keys=False
    ),
    encoding="utf-8",
)
print(f"Results YAML saved to {COMPARISON_RESULTS_PATH}")

Per-image results saved to data/wildreceipt/output/wildreceipt_internvl3_per_image.csv
Summary saved to data/wildreceipt/output/wildreceipt_internvl3_summary.json
Ground truth YAML saved to evaluation_data/wildreceipt_ground_truth.yaml
Results YAML saved to results/wildreceipt_internvl3_5_8b_results.yaml


---
## 13. Cleanup

In [13]:
# Remove the worker script written to disk
if _WORKER_SCRIPT.exists():
    _WORKER_SCRIPT.unlink()
    print(f"Removed {_WORKER_SCRIPT}")

gc.collect()
print("Cleanup complete.")

Removed data/wildreceipt/_wildreceipt_dp_worker.py
Cleanup complete.
